In [1]:
import os
import pathlib
import datetime as dt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

In [2]:
load_dotenv()

RAW_DIR = pathlib.Path(
    os.getenv("DATA_DIR_RAW", "data/raw")
)

PROC_DIR = pathlib.Path(
    os.getenv("DATA_DIR_PROCESSED", "data/processed")
)

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_DIR:", RAW_DIR.resolve())
print("PROC_DIR:", PROC_DIR.resolve())



RAW_DIR: C:\Users\schwa\bootcamp_michael_brick\homework\homework05\data\raw
PROC_DIR: C:\Users\schwa\bootcamp_michael_brick\homework\homework05\data\processed


In [3]:
np.random.seed(5)

dates = pd.date_range(
    "2024-01-01",
    periods=10,
    freq="D"
)

df = pd.DataFrame({
    "date": dates,
    "ticker": ["MRK"] * 10,
    "price": 150 + np.random.randn(10).cumsum()
})

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    10 non-null     datetime64[us]
 1   ticker  10 non-null     str           
 2   price   10 non-null     float64       
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 402.0 bytes


In [4]:
def ts():
    return dt.datetime.now().strftime("%Y%m%d-%H%M%S")

stamp = ts()

csv_path = RAW_DIR / f"sample_{stamp}.csv"
parq_path = PROC_DIR / f"sample_{stamp}.parquet"

df.to_csv(csv_path, index=False)
print("Saved CSV:", csv_path)

try:
    df.to_parquet(parq_path, index=False)
    print("Saved Parquet:", parq_path)

except Exception as e:
    print("Parquet save failed.")
    print("You may need to install pyarrow or fastparquet.")
    print("Error:", e)



Saved CSV: data\raw\sample_20260821-233808.csv
Saved Parquet: data\processed\sample_20260821-233808.parquet


In [5]:
df_csv = pd.read_csv(
    csv_path,
    parse_dates=["date"]
)

df_parq = pd.read_parquet(parq_path)

def validate_loaded(
    original: pd.DataFrame,
    reloaded: pd.DataFrame,
    cols=("date", "ticker", "price")
):
    checks = {
        "shape_equal": original.shape == reloaded.shape,
        "cols_present": all(
            c in reloaded.columns
            for c in cols
        )
    }

    if "price" in reloaded.columns:
        checks["price_is_numeric"] = (
            pd.api.types.is_numeric_dtype(
                reloaded["price"]
            )
        )

    if "date" in reloaded.columns:
        checks["date_is_datetime"] = (
            pd.api.types.is_datetime64_any_dtype(
                reloaded["date"]
            )
        )

    return checks

print(
    "CSV validation:",
    validate_loaded(df, df_csv)
)

print(
    "Parquet validation:",
    validate_loaded(df, df_parq)
)


CSV validation: {'shape_equal': True, 'cols_present': True, 'price_is_numeric': True, 'date_is_datetime': True}
Parquet validation: {'shape_equal': True, 'cols_present': True, 'price_is_numeric': True, 'date_is_datetime': True}


In [6]:
from typing import Union

def ensure_dir(path: pathlib.Path):
    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

def detect_format(path):
    suf = str(path).lower()

    if suf.endswith(".csv"):
        return "csv"

    if (
        suf.endswith(".parquet")
        or suf.endswith(".pq")
        or suf.endswith(".parq")
    ):
        return "parquet"

    raise ValueError(
        "Unsupported format for: " + str(path)
    )

In [7]:
def write_df(df, path):

    path = pathlib.Path(path)

    ensure_dir(path)

    fmt = detect_format(path)

    if fmt == "csv":
        df.to_csv(path, index=False)

    elif fmt == "parquet":
        try:
            df.to_parquet(path, index=False)

        except Exception as e:
            raise RuntimeError(
                "Parquet engine not available. "
                "Install pyarrow or fastparquet."
            ) from e

    return path

def read_df(path):

    path = pathlib.Path(path)

    fmt = detect_format(path)

    if fmt == "csv":

        columns = pd.read_csv(
            path,
            nrows=0
        ).columns

        if "date" in columns:
            return pd.read_csv(
                path,
                parse_dates=["date"]
            )

        return pd.read_csv(path)

    elif fmt == "parquet":

        try:
            return pd.read_parquet(path)

        except Exception as e:
            raise RuntimeError(
                "Parquet engine not available. "
                "Install pyarrow or fastparquet."
            ) from e

In [8]:
stamp2 = ts()

csv_util_path = RAW_DIR / f"sample_util_{stamp2}.csv"
pq_util_path = PROC_DIR / f"sample_util_{stamp2}.parquet"

write_df(df, csv_util_path)
write_df(df, pq_util_path)

df_csv_util = read_df(csv_util_path)
df_pq_util = read_df(pq_util_path)

print(
    "CSV utility validation:",
    validate_loaded(df, df_csv_util)
)

print(
    "Parquet utility validation:",
    validate_loaded(df, df_pq_util)
)

CSV utility validation: {'shape_equal': True, 'cols_present': True, 'price_is_numeric': True, 'date_is_datetime': True}
Parquet utility validation: {'shape_equal': True, 'cols_present': True, 'price_is_numeric': True, 'date_is_datetime': True}
